# AG_PRAXIS NB04 — Preprocessing and Splits

Everything after this notebook trains on what it writes. The capture files are read here
for the last time and turned into the arrays the later notebooks load: the records
themselves, cleaned and scaled, the partitions those records are divided into, and
sequences of consecutive records. Nothing after this opens a CSV.

Two things make that more than a change of file format.

The first is the split. Each class in this dataset was recorded in its own capture
session, and eight of the nineteen classes were recorded several times over. For those
eight I can keep whole recordings apart, so nothing a model trains on comes from the
session it is later tested on. For the other eleven there is one recording and no way to
do that, so the recording is cut into three consecutive blocks by row position instead.
Rows next to each other in a file can belong to the same burst of traffic, so a random
split would put related rows on both sides of the line and a test score would partly be
reading training data back. Contiguous blocks are the closest thing to holding a session
out that a single session allows, and what they still let through is stated further down
rather than left for a reader to work out.

The split as the dataset was distributed is built and saved as well. It is used for one
thing only, the reproduction of published work in NB05, which has to run on the same
partitions the published work ran on or it is not a reproduction.

The second is the scaler. It is fitted on training rows and no others. A scaler fitted on
everything has already read the test set, and every score after that is a slightly
generous account of data the model had not seen.

The whole thing runs twice. The fast pass builds both splits in full, checks them, and
cuts four windows from each block so the shapes and the arithmetic can be seen in a
couple of minutes. The full pass reads every row and writes the arrays the later
notebooks read. The last cells put the two passes side by side and print PASS or MISMATCH
for every value that cannot legitimately differ between them, and the ledger entry
refuses to describe itself as a reference run if any of them disagree.

Each pass writes to Drive: `splits.json`, the fitted scaler, one npz of records and one of
sequences for each of the three partitions, and `NB04_manifest.json` recording every
shape, every row count and a hash of every array, so a later notebook can confirm it
loaded the thing this notebook wrote.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from. Every array written
below belongs to that commit.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"
NOTEBOOK = "AG_PRAXIS_NB04_preprocessing_splits.ipynb"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

The paths, the seed, the split ratios, the window and the stride all come from
`config/base.yaml`. None of them is typed into a cell, so changing one is a commit rather
than an edit I later forget I made.

The two passes write to two different folders, `NB04_fast` and `NB04`, so neither can
overwrite the other. `NB04` is the full pass and is the one every later notebook reads.

In [ ]:
import json
import random
import time

import joblib
import numpy as np
import pandas as pd
import yaml

from src import captures as cap
from src import inventory as inv
from src import preprocessing as pre
from src import splits as sp

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB04_fast", "full": ARTIFACTS / "NB04"}

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )

ALL_FILES = sorted(TRAIN_DIR.glob("*.csv")) + sorted(TEST_DIR.glob("*.csv"))
if not ALL_FILES:
    raise FileNotFoundError(f"no CSV files under {TRAIN_DIR} or {TEST_DIR}")

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print(f"seed        : {SEED}")
print(f"train dir   : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir    : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"files       : {len(ALL_FILES)}")
print(f"split       : {CFG['split']['train']:.0%} train, {CFG['split']['val']:.0%} val, "
      f"{CFG['split']['test']:.0%} test")
print(f"sequences   : window {CFG['sequence']['window']}, stride {CFG['sequence']['stride']}")
print(f"fast pass   : {OUT_DIRS['fast']}")
print(f"full pass   : {OUT_DIRS['full']}   <- the one every later notebook reads")

The column list, the class list, the tier of each class and the number of rows in every
file were all settled by the inventory, which read all 8,775,013 rows. They are read from
the file it wrote rather than retyped here, and the row counts matter more than usual in
this notebook: the split is built from them before a single row is loaded, so a block can
be printed and checked while it is still only a range.

In [ ]:
INVENTORY_CANDIDATES = [
    REPO_ROOT / "data" / "processed" / "dataset_inventory.json",
    ARTIFACTS / "NB01" / "dataset_inventory.json",
]
INVENTORY_PATH = next((p for p in INVENTORY_CANDIDATES if p.exists()), None)
if INVENTORY_PATH is None:
    raise FileNotFoundError(
        "dataset_inventory.json not found. NB01 has to have been run, and its output moved "
        f"into data/processed/. Looked in: {[str(p) for p in INVENTORY_CANDIDATES]}"
    )

INVENTORY = json.loads(INVENTORY_PATH.read_text())
if INVENTORY.get("is_fast_pass"):
    raise ValueError(f"{INVENTORY_PATH} was written by NB01's fast pass and is not a result")

ALL_COLUMNS = list(INVENTORY["columns"])
CLASS_INFO = INVENTORY["classes"]
CLASSES = sorted(CLASS_INFO)
TIER = {label: CLASS_INFO[label]["tier"] for label in CLASSES}
TIER_A = sorted(INVENTORY["tiers"]["A"])
TIER_B = sorted(INVENTORY["tiers"]["B"])
ROWS_PER_FILE = {name: int(n) for name, n in INVENTORY["rows_per_file"].items()}
TOTAL_ROWS = int(INVENTORY["total_rows"])
CONSTANT_EVERYWHERE = list(INVENTORY["constant_columns"]["constant_everywhere"])

missing_counts = [p.name for p in ALL_FILES if p.name not in ROWS_PER_FILE]
if missing_counts:
    raise KeyError(f"files on Drive with no row count in the inventory: {missing_counts}")

print(f"inventory read from {INVENTORY_PATH}")
print(f"  written by   : {INVENTORY['generated_by']} at {INVENTORY['git_sha']} on "
      f"{INVENTORY['generated_on']}")
print(f"  rows scanned : {INVENTORY['rows_scanned']:,}")
print()
print(f"files            : {len(ALL_FILES)}")
print(f"columns          : {len(ALL_COLUMNS)}")
print(f"classes          : {len(CLASSES)}   tier A {len(TIER_A)}, tier B {len(TIER_B)}")
print(f"rows             : {TOTAL_ROWS:,}")
print(f"rows in the files: {sum(ROWS_PER_FILE[p.name] for p in ALL_FILES):,}")
print()
print(f"recorded more than once : {', '.join(TIER_A)}")
print(f"recorded once           : {', '.join(TIER_B)}")

assert len(CLASSES) == 19, f"expected 19 classes, got {len(CLASSES)}"
assert sum(ROWS_PER_FILE[p.name] for p in ALL_FILES) == TOTAL_ROWS, (
    "the row counts of the files on Drive do not add up to the inventory total"
)

Before anything is split I have to decide what a record is.

One column goes. The inventory found `Drate` holding the same value in every row of every
file, and a column that never changes separates nothing from anything, so forty-four
remain. `DHCP` stays. An earlier screen over the opening rows of each file reported that
column as constant too, but the head of a file is not the file, and reading all of it
showed `DHCP` moving.

Nothing else is dropped and nothing is recoded. In particular nothing is scaled yet. The
scaler comes after the split, because fitting it here would fit it on rows that are about
to become the test set.

In [ ]:
DROPPED_REASONS = {
    "Drate": "constant in every row of every file, so it separates nothing",
}

FEATURES, DROPPED = pre.select_features(ALL_COLUMNS, drop=DROPPED_REASONS)

print("dropped")
print(DROPPED.to_string(index=False))
print()
print(f"columns in the files : {len(ALL_COLUMNS)}")
print(f"dropped              : {len(DROPPED)}")
print(f"features kept        : {len(FEATURES)}")
print(f"DHCP kept            : {'DHCP' in FEATURES}")
print()
print("  " + "\n  ".join(", ".join(FEATURES[i : i + 6]) for i in range(0, len(FEATURES), 6)))

assert len(FEATURES) == 44, f"expected 44 features after the drop, got {len(FEATURES)}"
assert sorted(CONSTANT_EVERYWHERE) == sorted(DROPPED_REASONS), (
    f"the inventory found {CONSTANT_EVERYWHERE} constant everywhere and this cell drops "
    f"{sorted(DROPPED_REASONS)}"
)

One notebook downstream needs these features without the timing family, and this is where
that is settled.

The reason comes from inside this project. NB03 trained models to name the recording a
row came from rather than the attack it represents, and the four timing columns did that
better than all forty-four together, 0.9301 against 0.8010. Anything given those columns
can read the session, and an explanation read off such a model describes the recording
conditions as much as the attack. NB09 therefore explains a model trained without them.

What that notebook needs is a column slice of the sequences written here, and a slice is
not worth two and a half gigabytes of Drive and a second copy that can drift from the
first. So it is not written out. What is written instead is the definition: the manifest
records the four columns the timing family holds and the forty that remain, both by name
and in order, so `NB09` takes the slice by looking up those names in the feature list
saved beside the array. There is one array and one statement of what to drop from it,
rather than two arrays that have to be kept agreeing.

The families come from `config/feature_families.yaml`, which is marked reviewed. The
timing family in that file names five columns and one of them is `Drate`, which has just
been dropped, so the list is intersected with the columns that are actually live rather
than subtracted from them blindly. Subtracting a name that is not there would either
raise, or quietly remove nothing and leave a slice that is identical to the original
while still called something else.

In [ ]:
FAMILIES_PATH = REPO_ROOT / "config" / "feature_families.yaml"
FAMILY_DOC = yaml.safe_load(FAMILIES_PATH.read_text())

if FAMILY_DOC.get("review_required", True):
    raise ValueError(
        f"{FAMILIES_PATH} is still marked review_required. The family assignment decides "
        "which columns the timing-excluded variant loses, so it is read only once reviewed."
    )

TIMING_NAMED = list(FAMILY_DOC["families"]["timing"])
TIMING_LIVE = [c for c in FEATURES if c in set(TIMING_NAMED)]
FEATURES_NO_TIMING = pre.exclude_family(FEATURES, TIMING_NAMED)

print(f"read {FAMILIES_PATH}")
print(f"  status          : {FAMILY_DOC.get('status')}")
print(f"  review required : {FAMILY_DOC.get('review_required')}")
print(f"  reviewed on     : {FAMILY_DOC.get('reviewed_on')}")
print()
print(f"timing family as written  : {len(TIMING_NAMED)}   {', '.join(TIMING_NAMED)}")
print(f"of those still in the data: {len(TIMING_LIVE)}   {', '.join(TIMING_LIVE)}")
print(f"named but already dropped : {sorted(set(TIMING_NAMED) - set(TIMING_LIVE))}")
print()
print(f"features written with the sequences : {len(FEATURES)}")
print(f"features left after the slice       : {len(FEATURES_NO_TIMING)}   recorded in the "
      f"manifest, not written as a second array")

assert len(FEATURES_NO_TIMING) == len(FEATURES) - len(TIMING_LIVE)
assert not (set(FEATURES_NO_TIMING) & set(TIMING_LIVE)), "a timing column survived the exclusion"
assert len(TIMING_LIVE) == 4, f"expected four live timing columns, got {TIMING_LIVE}"

Next, what counts as a recording, because the split is built out of recordings and the
two tiers answer that question differently.

For the eight classes recorded more than once, a recording is one file, and its
identifier is the capture id with the partition attached. The partition has to be part of
it. The chunk numbers restart in each partition, so `TCP_IP-DDoS-ICMP1_test` is not a
second helping of `TCP_IP-DDoS-ICMP1_train`; it is a different session that happens to
carry the number one, and an identifier that dropped the partition would merge two
sessions into one.

For the other eleven the class has a single capture id, and that one id appears on both
sides of the distributed split. That is one session divided into two files rather than
two sessions, and the row counts show the division: the train file holds roughly four
fifths of the class and the test file the rest. So for those classes the recording is
both files together, train side first.

The order files are read in matters, because for tier B the split is by position within
that order. Files of a class are ordered train side first, then test side, then by chunk
number.

In [ ]:
RECORDINGS = sp.recording_table(ALL_FILES, ROWS_PER_FILE)

summary = (
    RECORDINGS.groupby("label")
    .agg(
        files=("file", "size"),
        capture_ids=("capture_id", "nunique"),
        rows=("n_rows", "sum"),
    )
    .reset_index()
)
summary["tier"] = summary["label"].map(TIER)
summary["recordings"] = np.where(summary["tier"] == "A", summary["files"], 1)

print(summary[["label", "tier", "capture_ids", "files", "recordings", "rows"]].to_string(index=False))
print()
print(RECORDINGS[["label", "order", "file", "shipped_partition", "chunk", "n_rows"]].to_string(index=False))
print()
print(f"files                       : {len(RECORDINGS)}")
print(f"recordings, tier A          : {int(summary.loc[summary['tier'] == 'A', 'recordings'].sum())}")
print(f"recordings, tier B          : {int(summary.loc[summary['tier'] == 'B', 'recordings'].sum())}")
print(f"rows                        : {int(RECORDINGS['n_rows'].sum()):,}")

for label in TIER_B:
    ids = RECORDINGS.loc[RECORDINGS["label"] == label, "capture_id"].nunique()
    assert ids == 1, f"{label} is in tier B and has {ids} capture ids, so it is not one recording"
for label in TIER_A:
    files = int((RECORDINGS["label"] == label).sum())
    assert files >= 3, f"{label} is in tier A with {files} files, too few to hold two out"
assert RECORDINGS["file"].is_unique, "two rows of the recording table claim the same file"
assert int(RECORDINGS["n_rows"].sum()) == TOTAL_ROWS

The seed is set before anything that could use it. Nothing here draws a random number:
the split is by position and by recording, the scaler is arithmetic, and the windows are
taken in order. That is deliberate, and the seed is set anyway so that a later change
which does introduce a choice cannot quietly make this notebook irreproducible.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
print(f"seeded with {SEED}")
print("nothing below samples, shuffles or draws; the seed is here so that nothing can")
print("start doing so unnoticed")

Everything below this point cuts sequences out of consecutive rows, fifty at a time,
moving on twenty-five rows between one window and the next. That only carries time if the
order of the rows in a file is the order the packets were recorded in. There is no
timestamp column in this dataset, so I cannot read the answer off the columns. It has to
be measured.

The measurement compares the file as it is against the same file with its order
destroyed. For each of the seventy-two files I read the first hundred thousand rows in
file order, or all of them if the file is smaller, and take the lag-1 autocorrelation of
three columns, which is how much each value resembles the value recorded before it. Then
I shuffle those same rows thirty times, with seeds zero to twenty-nine, and take the same
measurement on each shuffle. The shuffled runs are the null: what the number looks like
when position carries nothing. What I report is z, the distance between the real value
and the mean of that null, in standard deviations of the null.

Two of the three columns are timing, IAT and Rate. The third, Header_Length, is not, and
it is there as a control. If header size resembles the header size before it as strongly
as inter-arrival time does, then what I am measuring is not the passage of time, and I
would rather see that now than after building sequences on it.

The rule is fixed here, before the run. If z is above 3 for at least half of all file and
column combinations, row order carries structure and sequences of consecutive rows are
worth building. If fewer than half reach it, the premise does not hold, and the notebook
says so and stops before a single window is cut. A combination that cannot be scored at
all, because the column is constant or the file too short, counts against the premise
rather than being dropped from the count.

In [ ]:
ORDER_COLUMNS = ["IAT", "Rate", "Header_Length"]
ORDER_ROWS = 100_000
ORDER_SHUFFLES = 30
ORDER_Z = 3.0
ORDER_SHARE_REQUIRED = 0.5
ORDER_CHECK_PATH = OUT_DIRS["full"] / "row_order_check.json"

missing_order = [column for column in ORDER_COLUMNS if column not in ALL_COLUMNS]
if missing_order:
    raise KeyError(f"the row order check asks for columns the files do not have: {missing_order}")


def lag1(x):
    """How much each value resembles the one before it. NaN when there is too little to say."""
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size < 100 or np.std(x) == 0:
        return np.nan
    a, b = x[:-1], x[1:]
    return float(np.corrcoef(a, b)[0, 1])


def json_safe(value):
    """NaN is not JSON, so it is written as null."""
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    return value


order_rows = []
order_started = time.time()
print(f"reading the first {ORDER_ROWS:,} rows of each of {len(ALL_FILES)} files, "
      f"{len(ORDER_COLUMNS)} columns, {ORDER_SHUFFLES} shuffles each")

for i, path in enumerate(ALL_FILES, start=1):
    frame = pd.read_csv(path, usecols=ORDER_COLUMNS, nrows=ORDER_ROWS)
    label = cap.parse_capture(path.name)["label"]
    for column in ORDER_COLUMNS:
        values = frame[column].to_numpy(dtype=np.float64)
        observed = lag1(values)
        null = np.array(
            [lag1(np.random.default_rng(seed).permutation(values))
             for seed in range(ORDER_SHUFFLES)],
            dtype=np.float64,
        )
        usable = null[np.isfinite(null)]
        null_mean = float(usable.mean()) if usable.size else float("nan")
        null_std = float(usable.std(ddof=0)) if usable.size else float("nan")
        z = (observed - null_mean) / null_std if null_std > 0 else float("nan")
        order_rows.append(
            {
                "file": path.name,
                "label": label,
                "rows_read": int(len(frame)),
                "column": column,
                "observed": observed,
                "null_mean": null_mean,
                "null_std": null_std,
                "z": z,
            }
        )
    if i % 10 == 0 or i == len(ALL_FILES):
        print(f"  {i:>3}/{len(ALL_FILES)} files   {time.time() - order_started:6.1f}s")

ORDER = pd.DataFrame(order_rows)
ORDER["above"] = ORDER["z"] > ORDER_Z

order_table = (
    ORDER.groupby("column")
    .agg(
        median_observed=("observed", "median"),
        median_null=("null_mean", "median"),
        share_above_3=("above", "mean"),
    )
    .reindex(ORDER_COLUMNS)
    .reset_index()
)
assert all(
    pd.api.types.is_numeric_dtype(order_table[c])
    for c in ("median_observed", "median_null", "share_above_3")
), f"a column of the summary is not numeric:\n{order_table.dtypes}"

ORDER_SHARE_ABOVE = float(ORDER["above"].mean())
ORDER_VERDICT = "pass" if ORDER_SHARE_ABOVE >= ORDER_SHARE_REQUIRED else "fail"

print()
print(order_table.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print()
print(f"combinations         : {len(ORDER)}   {len(ALL_FILES)} files x {len(ORDER_COLUMNS)} columns")
print(f"above z of {ORDER_Z:.0f}        : {int(ORDER['above'].sum())}   {ORDER_SHARE_ABOVE:.1%}")
print(f"the rule needs       : {ORDER_SHARE_REQUIRED:.0%}")
print(f"could not be scored  : {int(ORDER['z'].isna().sum())}")
print()
if ORDER_VERDICT == "pass":
    print("Row order carries structure. Rows next to each other in a file resemble each")
    print("other more than shuffled rows do, so a window of consecutive rows holds")
    print("something a shuffled window does not, and sequences are worth building.")
else:
    print("Row order does not carry structure. A window of consecutive rows would hold")
    print("nothing a shuffled window does not, so the premise these sequences rest on")
    print("does not hold and the notebook stops below.")

ORDER_CHECK_PATH.parent.mkdir(parents=True, exist_ok=True)
ORDER_CHECK_PATH.write_text(
    json.dumps(
        json_safe(
            {
                "generated_by": NOTEBOOK,
                "git_sha": GIT_SHA,
                "generated_on": RUN_DATE,
                "columns": ORDER_COLUMNS,
                "control_column": "Header_Length",
                "rows_read_per_file": ORDER_ROWS,
                "shuffles": ORDER_SHUFFLES,
                "shuffle_seeds": list(range(ORDER_SHUFFLES)),
                "z_threshold": ORDER_Z,
                "share_required": ORDER_SHARE_REQUIRED,
                "rule": (
                    "row order carries structure if z is above 3 for at least half of all "
                    "file and column combinations"
                ),
                "files": len(ALL_FILES),
                "combinations": int(len(ORDER)),
                "combinations_above_z": int(ORDER["above"].sum()),
                "combinations_not_scored": int(ORDER["z"].isna().sum()),
                "share_above_z": ORDER_SHARE_ABOVE,
                "per_column": order_table.to_dict(orient="records"),
                "per_file": order_rows,
                "verdict": ORDER_VERDICT,
            }
        ),
        indent=2,
    )
)

The check is on Drive whichever way it went, so a failure leaves the numbers behind
rather than an empty folder. This cell is the stop. It raises if fewer than half the
combinations reached z of 3, and nothing below it runs.

In [ ]:
if ORDER_VERDICT != "pass":
    raise RuntimeError(
        f"row order does not carry structure: {ORDER_SHARE_ABOVE:.1%} of file and column "
        f"combinations reach z above {ORDER_Z:.0f}, against the {ORDER_SHARE_REQUIRED:.0%} "
        "the rule needs. Windows of consecutive rows would carry no more than windows of "
        f"shuffled rows. The numbers are in {ORDER_CHECK_PATH}."
    )

print(f"row order check passed: {ORDER_SHARE_ABOVE:.1%} of combinations above z of {ORDER_Z:.0f}")
print(f"written: {ORDER_CHECK_PATH}")

What follows is one function per step, then a driver that calls them in order, then the
cell that runs the whole thing twice. Every step prints its own results and returns them,
so nothing is recomputed later and the comparison at the end has something to compare.

This cell holds what the two passes share. The window is fifty records and the stride
twenty-five, both from the config file, so consecutive windows overlap by half and every
row inside a block appears in some window.

The only thing the fast pass changes is a cap of four windows per block, and through that
how many rows get read at all. Four windows need a hundred and twenty-five rows, so the
fast pass reads a hundred and twenty-five rows from each block and the full pass reads
every row of every block. Nothing is computed a different way, and both passes use the
same window, the same stride and the same scaler class.

The arrays are written as float32. That is what the models will consume and it halves
what has to travel to and from Drive, which matters here more than in any other notebook
because of how much of it there is.

In [ ]:
RATIOS = (CFG["split"]["train"], CFG["split"]["val"], CFG["split"]["test"])
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARRAY_DTYPE = np.float32

FAST_WINDOWS_PER_BLOCK = 4
WINDOWS_PER_BLOCK = {"fast": FAST_WINDOWS_PER_BLOCK, "full": None}
ROWS_PER_BLOCK = {"fast": WINDOW + STRIDE * (FAST_WINDOWS_PER_BLOCK - 1), "full": None}

SCALER_PREVIEW = 8
PROGRESS_EVERY = 10


def section(title):
    print()
    print("-" * 79)
    print(title)
    print("-" * 79)


def banner(lines):
    print()
    print("#" * 79)
    for line in lines:
        print(f"#  {line:<75}#")
    print("#" * 79)


def human(n_bytes):
    value = float(n_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if value < 1024.0 or unit == "GB":
            return f"{value:,.1f} {unit}"
        value /= 1024.0


def merge(results, part):
    """Fold a step's return value into the run, keeping what every step wrote."""
    results["written"] = results.get("written", []) + list(part.pop("written", []))
    results.update(part)
    return results


def reader_progress(what, every=PROGRESS_EVERY):
    """A callback that prints one line every `every` blocks, and on the last."""
    started = time.time()

    def report(i, total, block, *extra):
        if i % every and i != total:
            return
        detail = f"{extra[0]:>9,} rows{extra[1]:>8,} windows" if extra else ""
        print(f"  {what:<9} block {i:>3}/{total}   {block['file'][:40]:<40} {detail}"
              f"   {time.time() - started:6.1f}s")

    return report


print(f"split ratios        : {RATIOS[0]:.0%} / {RATIOS[1]:.0%} / {RATIOS[2]:.0%}")
print(f"window, stride      : {WINDOW} records, {STRIDE} records")
print(f"overlap             : {WINDOW - STRIDE} records between consecutive windows")
print(f"array dtype         : {np.dtype(ARRAY_DTYPE)}")
print(f"windows per block   : fast {WINDOWS_PER_BLOCK['fast']}, full uncapped")
print(f"rows read per block : fast {ROWS_PER_BLOCK['fast']}, full every row")
print(f"written per partition: one record array and one sequence array, both {len(FEATURES)} "
      f"features")

assert STRIDE <= WINDOW, "a stride wider than the window would skip rows entirely"

Step one, the two splits.

The shipped split is the one the dataset arrived with: every file whole, on the side its
name gives. It is saved because NB05 reproduces published results and has to use the
partitions those results were produced on.

The two-tier split is the one everything else uses. For a class recorded several times,
the earliest recordings train, the next to last validates, and the last tests, each one
whole. For a class recorded once, the single recording is cut at 70% and 85% of its
length and the three stretches become the three partitions. Where that recording arrived
as two files the positions run through the first file and then the second, so a cut can
land inside either of them, and a class can have a validation range that ends in one file
and continues into the next.

Both are built from row counts alone, so this step reads nothing.

In [ ]:
def build_splits(fast: bool) -> dict:
    section("Step 1 - the two splits")

    shipped = sp.shipped_blocks(RECORDINGS, TIER)
    two_tier = sp.two_tier_blocks(RECORDINGS, TIER, ratios=RATIOS)

    shipped_rows = sp.rows_per_partition(shipped)
    print("The shipped split. Whole files, nothing moved.")
    for name, count in shipped_rows.items():
        files = sorted({b["file"] for b in shipped if b["partition"] == name})
        print(f"  {name:<6} {len(files):>2} files {count:>10,} rows {count / TOTAL_ROWS:>7.2%}")

    print()
    print("The two-tier split. Every block, and for each one the recording or the row range")
    print("it takes.")
    print()
    block_table = sp.describe_blocks(two_tier)
    print(block_table.to_string(index=False))

    frame = sp.blocks_frame(two_tier)
    two_tier_rows = sp.rows_per_partition(two_tier)
    asked = dict(zip(sp.PARTITION_ORDER, RATIOS))
    print()
    print(f"{'partition':<11}{'rows':>12}{'share':>9}{'asked':>8}{'recordings':>13}{'blocks':>8}")
    for name in sp.PARTITION_ORDER:
        part = frame[frame["partition"] == name]
        print(f"{name:<11}{two_tier_rows[name]:>12,}{two_tier_rows[name] / TOTAL_ROWS:>9.2%}"
              f"{asked[name]:>8.0%}{part['recording'].nunique():>13}{len(part):>8}")

    print()
    print("The shares are not exactly what was asked for and cannot be. A tier A class hands")
    print("over whole recordings, so it contributes whatever those recordings happen to hold.")

    return {
        "shipped_blocks": shipped,
        "two_tier_blocks": two_tier,
        "shipped_rows": shipped_rows,
        "two_tier_rows": two_tier_rows,
        "block_table": block_table,
        "block_signature": frame[
            ["label", "tier", "partition", "recording", "file", "start", "stop"]
        ].to_dict(orient="records"),
        "shipped_signature": sp.blocks_frame(shipped)[
            ["label", "partition", "file", "start", "stop"]
        ].to_dict(orient="records"),
    }

Step two, whether the split is sound. Four things have to be true before any of it is
worth writing.

No recording appears in more than one partition, for the eight classes where that is
possible at all. Every class appears in all three partitions, because a class missing
from validation or test cannot be scored. The class proportions of each partition are
printed so a class that is a tenth of training and a fiftieth of test is visible rather
than discovered later in a confusion matrix. And the row counts add up to the dataset
total, which is the check that catches a block quietly dropped or counted twice.

Two more are checked because they are cheap: that the blocks of each file cover it
exactly once with no gap and no overlap, and that no block came out empty. If any check
fails the notebook stops here and writes nothing.

In [ ]:
def check_splits(r: dict, fast: bool) -> dict:
    section("Step 2 - is the split sound")

    checks = {}
    for protocol, blocks, partitions in (
        ("shipped", r["shipped_blocks"], ["train", "test"]),
        ("two_tier", r["two_tier_blocks"], list(sp.PARTITION_ORDER)),
    ):
        table = sp.validate_split(
            blocks,
            classes=CLASSES,
            expected_partitions=partitions,
            total_rows=TOTAL_ROWS,
            rows_per_file=ROWS_PER_FILE,
            disjoint_classes=TIER_A,
        )
        checks[protocol] = table
        print(f"{protocol} split")
        print(table.to_string(index=False))
        print()

    proportions = sp.class_proportions(r["two_tier_blocks"])
    print("Class proportions in the two-tier split. `rows` is the class total; the three")
    print("percentages are that class's share of each partition.")
    print(proportions.to_string(index=False))

    drift = (proportions["train_pct"] - proportions["test_pct"]).abs()
    worst = proportions.loc[drift.idxmax()]
    print()
    print(f"largest difference between a class's share of training and of test: "
          f"{worst['label']}, {worst['train_pct']:.2f}% against {worst['test_pct']:.2f}%")
    print("The tier A classes move most, which is the price of holding whole recordings out.")

    failed = {
        protocol: table.loc[table["result"] == "FAIL", "check"].tolist()
        for protocol, table in checks.items()
    }
    failed = {protocol: names for protocol, names in failed.items() if names}
    if failed:
        raise RuntimeError(
            f"the split does not hold: {failed}. Nothing is written and nothing below this "
            "point should be read as a result."
        )

    print()
    print("Every check passed, so there is something worth writing.")

    return {
        "checks": checks,
        "checks_signature": {p: t.to_dict(orient="records") for p, t in checks.items()},
        "class_proportions": proportions,
    }

Step three says what the two tiers mean, because a split that treats two groups of
classes differently has to be read differently for each of them.

For the eight classes recorded more than once, a model trains on one set of sessions and
is tested on a session it has never seen. A score on those classes is a statement about
new recordings.

For the eleven classes recorded once, the three partitions are three consecutive
stretches of a single session. Everything constant about that session, the machine, the
link, the load on it, the moment it was made, is present on both sides of the split. A
model can use it, and the test score for those classes is not evidence that the same
model would work on a recording made another day.

That is a limitation and not a bug, and it is recorded rather than fixed, because fixing
it needs a second recording of those classes and there is not one.

In [ ]:
def state_limitation(r: dict, fast: bool) -> dict:
    section("Step 3 - what the two tiers mean")

    frame = sp.blocks_frame(r["two_tier_blocks"])
    per_class = (
        frame.pivot_table(index=["tier", "label"], columns="partition", values="n_rows",
                          aggfunc="sum")
        .reindex(columns=list(sp.PARTITION_ORDER))
        .fillna(0)
        .astype("int64")
        .reset_index()
    )
    within = per_class[per_class["tier"] == "B"]
    across = per_class[per_class["tier"] == "A"]

    print("Recordings held out whole, so training and test share no session:")
    print(across.to_string(index=False))
    print()
    print("One recording cut into three stretches, so training and test share a session:")
    print(within.to_string(index=False))
    print()

    rows_within = int(frame.loc[frame["tier"] == "B", "n_rows"].sum())
    test_within = int(
        frame.loc[(frame["tier"] == "B") & (frame["partition"] == "test"), "n_rows"].sum()
    )
    test_rows = int(frame.loc[frame["partition"] == "test", "n_rows"].sum())

    statement = (
        f"{len(TIER_B)} of {len(CLASSES)} classes have one recording each. Their training, "
        "validation and test rows are consecutive stretches of that single recording, so "
        "anything a model learns about the session helps it on all three sides. Their scores "
        "are not evidence of generalisation to a new recording."
    )
    print(statement)
    print()
    print(f"classes split within one recording : {len(TIER_B)} of {len(CLASSES)}")
    print(f"  {', '.join(TIER_B)}")
    print(f"rows in those classes              : {rows_within:,} of {TOTAL_ROWS:,} "
          f"({rows_within / TOTAL_ROWS:.2%})")
    print(f"their share of the test partition  : {test_within:,} of {test_rows:,} "
          f"({test_within / test_rows:.2%})")
    print()
    print("The eleven are also the small classes, so they are a fortieth of the rows and more")
    print("than half of the classes. Macro-F1 weights every class the same, which means the")
    print("primary metric of this project leans on exactly the classes this limitation")
    print("applies to. That is worth saying out loud rather than filing away.")

    return {
        "limitation": {
            "classes_split_within_one_recording": list(TIER_B),
            "n_classes": len(TIER_B),
            "of_classes": len(CLASSES),
            "rows": rows_within,
            "share_of_rows": round(rows_within / TOTAL_ROWS, 6),
            "share_of_test_partition": round(test_within / test_rows, 6),
            "statement": statement,
        },
        "rows_by_class": per_class,
    }

Step four fits the scaler.

It sees training blocks and nothing else, which is the whole of the guarantee: the
function is handed a list of blocks and cannot read a row that is not in one of them. It
is fitted one block at a time, so the mean and variance are accumulated across the
partition without holding it in memory, and the result is identical to fitting on the
whole training partition at once.

What comes back is checked three ways: that every block it read was a training block,
that none of the validation or test blocks was touched, and that the number of rows it
saw is exactly the number the training blocks hold.

In [ ]:
def fit_the_scaler(r: dict, fast: bool) -> dict:
    section("Step 4 - the scaler, fitted on training rows only")

    mode = "fast" if fast else "full"
    cap = ROWS_PER_BLOCK[mode]
    train_blocks = [b for b in r["two_tier_blocks"] if b["partition"] == "train"]
    held_out = {
        (b["file"], int(b["start"]))
        for b in r["two_tier_blocks"]
        if b["partition"] != "train"
    }
    train_rows = sum(b["n_rows"] for b in train_blocks)
    expected = sum(min(b["n_rows"], cap) if cap else b["n_rows"] for b in train_blocks)

    print(f"{len(train_blocks)} training blocks holding {train_rows:,} rows")
    print(f"reading {'every row' if cap is None else f'the first {cap} rows of each block'}, "
          f"{expected:,} rows in total")
    print()

    started = time.time()
    scaler, used = pre.fit_scaler(
        train_blocks, FEATURES, max_rows_per_block=cap, progress=reader_progress("scaler")
    )
    elapsed = time.time() - started

    seen = int(np.ravel(scaler.n_samples_seen_)[0])
    partitions_used = sorted(set(used["partition"]))
    touched_held_out = {(row.file, int(row.start)) for row in used.itertuples()} & held_out

    print()
    print(f"blocks read                      : {len(used)} of {len(train_blocks)}")
    print(f"partitions those blocks were in  : {partitions_used}")
    print(f"validation or test blocks touched: {len(touched_held_out)}")
    print(f"rows the scaler saw              : {seen:,}")
    print(f"rows it was supposed to see      : {expected:,}")
    print(f"time                             : {elapsed:.1f}s")
    print()
    statistics = pre.scaler_statistics(scaler, FEATURES)
    print(f"the first {SCALER_PREVIEW} features, as fitted")
    print(statistics.head(SCALER_PREVIEW).to_string(index=False))

    assert partitions_used == ["train"], f"the scaler read {partitions_used}, not train alone"
    assert not touched_held_out, f"the scaler read held-out blocks: {sorted(touched_held_out)}"
    assert len(used) == len(train_blocks), "the scaler skipped a training block"
    assert seen == expected, f"the scaler saw {seen:,} rows and {expected:,} were expected"
    assert np.isfinite(scaler.mean_).all() and np.isfinite(scaler.scale_).all()
    print()
    print("Fitted on training rows only, confirmed against the block list rather than assumed.")

    return {
        "scaler": scaler,
        "scaler_blocks": used,
        "scaler_signature": used[["partition", "label", "file", "start"]].to_dict(
            orient="records"
        ),
        "scaler_rows": seen,
        "scaler_train_only": partitions_used == ["train"] and not touched_held_out,
        "scaler_statistics": statistics,
        "scaler_seconds": elapsed,
    }

Step five reads the rows and writes the arrays.

Each block is read once and gives up two things. The records are its rows, scaled, in the
order they appear in the file. The sequences are windows of fifty consecutive rows cut
from those same rows every twenty-five, and a window never crosses a file boundary
because it is cut inside a block and a block never does. Reading once for both is not
only cheaper, it is what makes them the same numbers: there is no second read that could
pick up a different scaler or a different slice.

The rows are ordered by their position in the file. There is no timestamp column in this
dataset, so position is the only ordering there is, and a sequence is fifty rows in the
order the extractor wrote them. Each sequence takes the label of the file it came from,
which is the class of that whole recording.

How many rows and how many windows each block gives is worked out from the row ranges
before anything is read, so the arrays are allocated once at their final size and the
totals below can be compared against what actually came out. A block shorter than fifty
rows yields no window and still keeps its rows in the record array; how many such blocks
there are is reported rather than silently absorbed.

Each partition is written as soon as it is built, before the next one starts. Holding all
three in memory at once is the thing that would fail.

In [ ]:
def build_and_write_arrays(r: dict, fast: bool) -> dict:
    section("Step 5 - records and sequences")

    mode = "fast" if fast else "full"
    cap = WINDOWS_PER_BLOCK[mode]
    out_dir = r["out_dir"]

    implied, plans = {}, {}
    for name in sp.PARTITION_ORDER:
        blocks = [b for b in r["two_tier_blocks"] if b["partition"] == name]
        whole = sp.build_plan(blocks, window=WINDOW, stride=STRIDE)
        plans[name] = sp.build_plan(blocks, window=WINDOW, stride=STRIDE, cap=cap)
        short = whole[whole["n_available"] == 0]
        implied[name] = {
            "n_blocks": int(len(whole)),
            "n_rows": int(whole["n_rows"].sum()),
            "n_sequences": int(whole["n_available"].sum()),
            "blocks_shorter_than_a_window": int(len(short)),
            "rows_in_those_blocks": int(short["n_rows"].sum()),
            "rows_past_the_last_window": int((whole["n_rows"] - whole["rows_used"]).sum()),
            "sequences_by_class": {
                str(k): int(v) for k, v in whole.groupby("label")["n_available"].sum().items()
            },
        }

    bytes_sequences = sum(
        implied[name]["n_sequences"] * WINDOW * len(FEATURES) * np.dtype(ARRAY_DTYPE).itemsize
        for name in sp.PARTITION_ORDER
    )
    bytes_records = sum(
        implied[name]["n_rows"] * len(FEATURES) * np.dtype(ARRAY_DTYPE).itemsize
        for name in sp.PARTITION_ORDER
    )

    print("What the split implies, before anything is read:")
    print()
    print(f"{'partition':<11}{'blocks':>8}{'rows':>12}{'windows':>10}{'short blocks':>14}"
          f"{'rows past last window':>23}")
    for name in sp.PARTITION_ORDER:
        line = implied[name]
        print(f"{name:<11}{line['n_blocks']:>8}{line['n_rows']:>12,}{line['n_sequences']:>10,}"
              f"{line['blocks_shorter_than_a_window']:>14}"
              f"{line['rows_past_the_last_window']:>23,}")
    print()
    print(f"{'records':<32}{human(bytes_records)}")
    print(f"{'sequences':<32}{human(bytes_sequences)}")
    print(f"{'total if every row is read':<32}{human(bytes_records + bytes_sequences)}")
    if cap is not None:
        print(f"this pass caps each block at {cap} windows, so it writes a small fraction of that")

    array_summary, counts, written = {}, [], []
    for name in sp.PARTITION_ORDER:
        plan = plans[name]
        print()
        print(f"{name}: {int(plan['rows_to_read'].sum()):,} rows to read, "
              f"{int(plan['n_sequences'].sum()):,} windows to cut")
        built = pre.build_partition(
            plan,
            FEATURES,
            r["scaler"],
            window=WINDOW,
            stride=STRIDE,
            dtype=ARRAY_DTYPE,
            progress=reader_progress(name),
        )

        row_counts = pre.counts_by_class(built, "records").rename(columns={"n": "rows"})
        sequence_counts = pre.counts_by_class(built, "sequences").rename(
            columns={"n": "sequences"}
        )
        counted = row_counts.merge(sequence_counts, on="label")
        counted.insert(0, "partition", name)
        counts.append(counted)

        records = built.pop("records")
        path = pre.save_arrays(out_dir / f"records_{name}.npz", built, "records", records)
        written.append(pre.describe_written(path, arrays=records))
        array_summary[f"records_{name}"] = {
            "file": path.name,
            "shape": list(records["X"].shape),
            "n_features": len(FEATURES),
            "n_rows": int(records["X"].shape[0]),
            "by_class": dict(zip(row_counts["label"], row_counts["rows"].astype(int))),
        }
        print(f"  records   {str(records['X'].shape):>22}  -> {path.name}")
        del records

        sequences = built["sequences"]
        path = pre.save_arrays(out_dir / f"sequences_{name}.npz", built, "sequences")
        written.append(pre.describe_written(path, arrays=sequences))
        array_summary[f"sequences_{name}"] = {
            "file": path.name,
            "shape": list(sequences["X"].shape),
            "n_features": len(FEATURES),
            "window": WINDOW,
            "stride": STRIDE,
            "by_class": dict(
                zip(sequence_counts["label"], sequence_counts["sequences"].astype(int))
            ),
        }
        print(f"  sequences {str(sequences['X'].shape):>22}  -> {path.name}")
        assert sequences["X"].shape[2] == len(FEATURES)
        del sequences, built

    counts = pd.concat(counts, ignore_index=True)
    table = counts.pivot(index="label", columns="partition", values="sequences").reindex(
        columns=list(sp.PARTITION_ORDER)
    )
    print()
    print("sequences per class per partition")
    print(table.to_string())

    rows_built = {
        name: array_summary[f"records_{name}"]["n_rows"] for name in sp.PARTITION_ORDER
    }
    print()
    print(f"{'partition':<11}{'rows in the split':>20}{'rows written':>15}{'windows written':>18}")
    for name in sp.PARTITION_ORDER:
        built_sequences = array_summary[f"sequences_{name}"]["shape"][0]
        print(f"{name:<11}{r['two_tier_rows'][name]:>20,}{rows_built[name]:>15,}"
              f"{built_sequences:>18,}")

    if cap is None:
        for name in sp.PARTITION_ORDER:
            assert rows_built[name] == r["two_tier_rows"][name], (
                f"{name}: the split holds {r['two_tier_rows'][name]:,} rows and "
                f"{rows_built[name]:,} were written"
            )
        assert sum(rows_built.values()) == TOTAL_ROWS
        print()
        print("Every row of the dataset is in exactly one record array.")

    empty = [
        (str(label), str(name))
        for label in table.index
        for name in table.columns
        if pd.isna(table.loc[label, name]) or int(table.loc[label, name]) == 0
    ]
    assert not empty, f"a class produced no sequences in some partition: {empty}"
    print()
    print(f"Every one of the {len(table)} classes produced sequences in all three partitions, "
          f"the fewest being {int(table.min().min())}.")

    return {
        "implied": implied,
        "array_summary": array_summary,
        "sequence_counts": counts,
        "sequence_table": table,
        "rows_built": rows_built,
        "written": written,
    }

Step six writes the three things that describe the rest.

`splits.json` holds both protocols as plain data: for each partition, which files it
uses, which recordings, and the exact row range taken from each. A later notebook can
read it and know what is on each side without rebuilding any of the reasoning here.

The shipped split is saved as those file lists and row ranges and nothing else. No arrays
are written for it, and that is on purpose. NB05 reproduces the published baseline, which
means running it as published: its own preprocessing, and a scaler fitted on the shipped
training side. Handing it rows scaled by the scaler fitted above would put this project's
preprocessing inside a result whose whole value is that it was produced the way the
original was, and the comparison it is there to provide would mean nothing. So NB05 reads
the CSVs itself, using this file to know which ones and which rows. It is the one
notebook that opens a CSV, and that is deliberate rather than an omission here.

The scaler is saved as it was fitted, so a notebook that needs to transform a new row
uses the same means and the same scales rather than fitting its own.

The manifest records what every array came out as, how many rows and sequences each
partition holds, how many of each class, and a hash of every array taken before it was
written. Hashing the array rather than the file it went into means a later notebook can
load an npz, hash what it got, and confirm it is holding this run's output.

In [ ]:
def write_manifest(r: dict, fast: bool) -> dict:
    section("Step 6 - splits, scaler and manifest")

    mode = "fast" if fast else "full"
    out_dir = r["out_dir"]
    written = []
    two_tier_frame = sp.blocks_frame(r["two_tier_blocks"])

    splits_document = {
        "shipped": sp.split_document(r["shipped_blocks"], RECORDINGS, protocol="shipped"),
        "two_tier": sp.split_document(
            r["two_tier_blocks"], RECORDINGS, protocol="two_tier", ratios=RATIOS
        ),
    }
    splits_path = out_dir / "splits.json"
    splits_path.write_text(json.dumps(cap.jsonable(splits_document), indent=2) + "\n")
    written.append(pre.describe_written(splits_path))

    scaler_path = out_dir / "scaler.joblib"
    joblib.dump(r["scaler"], scaler_path)
    written.append(pre.describe_written(scaler_path))

    statistics = r["scaler_statistics"]
    manifest = {
        "generated_by": NOTEBOOK,
        "generated_on": RUN_DATE,
        "git_sha": GIT_SHA,
        "git_dirty": GIT_DIRTY,
        "seed": SEED,
        "mode": mode,
        "is_fast_pass": bool(fast),
        "enterable_in_ledger": not fast,
        "read_from": {
            "inventory": str(INVENTORY_PATH),
            "families": str(FAMILIES_PATH),
            "n_files": len(ALL_FILES),
            "total_rows": TOTAL_ROWS,
        },
        "columns": {
            "in_the_files": len(ALL_COLUMNS),
            "dropped": DROPPED_REASONS,
            "kept": FEATURES,
            "timing_family_named": TIMING_NAMED,
            "timing_family_live": TIMING_LIVE,
        },
        "timing_excluded_slice": {
            "why": (
                "NB03 measured 0.9301 capture-identification accuracy from the timing family "
                "alone against 0.8010 from all 44 features, so NB09 explains a model trained "
                "without it"
            ),
            "not_written": (
                "a column slice of the sequence arrays, recorded here rather than saved as a "
                "second copy that could drift from the first"
            ),
            "drop": TIMING_LIVE,
            "keep": FEATURES_NO_TIMING,
            "n_features": len(FEATURES_NO_TIMING),
            "how": (
                "index = [list(npz['features']).index(c) for c in manifest['timing_excluded_"
                "slice']['keep']]; X[:, :, index]"
            ),
        },
        "sequences": {
            "window": WINDOW,
            "stride": STRIDE,
            "dtype": str(np.dtype(ARRAY_DTYPE)),
            "windows_per_block_cap": WINDOWS_PER_BLOCK[mode],
            "row_order": "position in the file; this dataset has no timestamp column",
            "label_rule": "a sequence takes the label of the file it was cut from",
        },
        "split": {
            "protocol": "two_tier",
            "ratios": list(RATIOS),
            "rows_per_partition": r["two_tier_rows"],
            "recordings_per_partition": {
                name: int(two_tier_frame.loc[two_tier_frame["partition"] == name, "recording"]
                          .nunique())
                for name in sp.PARTITION_ORDER
            },
            "checks": r["checks_signature"]["two_tier"],
            "blocks": r["block_signature"],
        },
        "shipped_split": {
            "rows_per_partition": r["shipped_rows"],
            "checks": r["checks_signature"]["shipped"],
            "used_for": "reproduction of published results in NB05, and nothing else",
            "arrays_written": False,
            "why_no_arrays": (
                "NB05 reproduces the published baseline as published, fitting its own scaler "
                "on the shipped training side. Rows scaled by this notebook's scaler would put "
                "this project's preprocessing inside a result whose value is that it was "
                "produced the way the original was. NB05 reads the CSVs itself, using the file "
                "lists and row ranges in splits.json, and is the only notebook that does."
            ),
        },
        "limitation": r["limitation"],
        "scaler": {
            "fitted_on": "train",
            "n_blocks": int(len(r["scaler_blocks"])),
            "n_rows": int(r["scaler_rows"]),
            "file": scaler_path.name,
            "mean": dict(zip(statistics["feature"], statistics["mean"].round(6))),
            "scale": dict(zip(statistics["feature"], statistics["scale"].round(6))),
        },
        "arrays": r["array_summary"],
        "files": r["written"] + written,
    }

    manifest_path = out_dir / "NB04_manifest.json"
    manifest_path.write_text(json.dumps(cap.jsonable(manifest), indent=2, default=str) + "\n")
    written.append(pre.describe_written(manifest_path))

    print(f"{splits_path.name:<34}{human(splits_path.stat().st_size):>12}   "
          f"{splits_document['two_tier']['n_blocks']} blocks in the two-tier split, "
          f"{splits_document['shipped']['n_blocks']} in the shipped one")
    print(f"{scaler_path.name:<34}{human(scaler_path.stat().st_size):>12}   "
          f"fitted on {r['scaler_rows']:,} training rows")
    print(f"{manifest_path.name:<34}{human(manifest_path.stat().st_size):>12}   "
          f"{len(manifest['files'])} files described")

    return {"manifest": manifest, "manifest_path": manifest_path,
            "splits_path": splits_path, "scaler_path": scaler_path, "written": written}

`run_preprocessing` calls the steps in order and hands each one what the previous ones
produced. It takes a single argument, and the only thing that argument changes is how
many windows are taken from each block and therefore how many rows are read.

The arrays are dropped from the returned result as soon as they are written, so two
passes' worth of data are never in memory at once. Everything computed from them, the
shapes, the counts and the hashes, is kept.

In [ ]:
def run_preprocessing(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    out_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    r = {"mode": mode, "fast": fast, "out_dir": out_dir, "written": []}

    merge(r, build_splits(fast))
    merge(r, check_splits(r, fast))
    merge(r, state_limitation(r, fast))
    merge(r, fit_the_scaler(r, fast))
    merge(r, build_and_write_arrays(r, fast))
    merge(r, write_manifest(r, fast))

    r["elapsed_s"] = time.time() - started
    r["bytes_written"] = sum(entry["bytes"] for entry in r["written"])

    section(f"What the {mode} pass wrote to {out_dir}")
    for entry in r["written"]:
        described = entry.get("arrays", {}).get("X")
        shape = str(tuple(described["shape"])) if described else ""
        print(f"  {entry['file']:<34}{human(entry['bytes']):>12}   {shape}")
    print(f"  {'total':<34}{human(r['bytes_written']):>12}")
    if IN_COLAB:
        print()
        print("Colab cannot push to the repository from a cell. The manifest and splits.json")
        print("are downloaded from Drive and moved into data/processed/ from the Mac. The")
        print("arrays stay on Drive; they are far too large for git and every later notebook")
        print("reads them from there.")

    return r

Both passes run here, fast first. If the fast one raises, the full one never starts,
which is the point: a wrong path or a mistake in the arithmetic costs a couple of minutes
rather than the length of a full read of the dataset.

In [ ]:
BANNERS = {
    "fast": [
        "FAST PASS",
        "both splits in full, every check, and four windows per block so that",
        "the shapes and the arithmetic can be seen quickly",
        "not a result, and never entered in the ledger",
    ],
    "full": [
        "FULL PASS",
        "every row of every file read once, every window cut",
        "this is the pass every later notebook reads",
    ],
}

results = {}
for fast in (True, False):
    name = "fast" if fast else "full"
    banner(BANNERS[name])
    results[name] = run_preprocessing(fast)

FAST, FULL = results["fast"], results["full"]
banner([f"fast pass {FAST['elapsed_s']:.0f}s, full pass {FULL['elapsed_s']:.0f}s",
        f"full pass wrote {human(FULL['bytes_written'])} to {FULL['out_dir']}"])

The full pass's manifest, printed whole, and the checks from step two again underneath
it. Colab cannot push to the repository from a cell, so until the artefacts are moved
across by hand the saved copy of this notebook is the only durable record of what the run
produced. Printing the manifest means the executed notebook carries it even if the Drive
folder is later cleared.

Only the full pass is printed. The fast pass's copies are under `NB04_fast` if they are
ever wanted for debugging, and they are not results.

In [ ]:
print("=" * 79)
print(f"{FULL['manifest_path']}   ({FULL['manifest_path'].stat().st_size:,} bytes)")
print("=" * 79)
print(FULL["manifest_path"].read_text().rstrip())
print()

print("=" * 79)
print(f"{FULL['splits_path']}   ({FULL['splits_path'].stat().st_size:,} bytes)")
print("=" * 79)
for protocol, document in json.loads(FULL["splits_path"].read_text()).items():
    print(f"{protocol}: {document['n_blocks']} blocks over {document['n_files']} files, "
          f"{document['n_rows']:,} rows")
    for name, part in document["partitions"].items():
        print(f"  {name:<6}{part['n_rows']:>12,} rows  {len(part['files']):>2} files  "
              f"{len(part['recordings']):>2} recordings  {part['n_blocks']:>2} blocks")
print("The block by block listing is in the file and was printed in full in step one.")
print()

print("=" * 79)
print("the checks, from the full pass")
print("=" * 79)
for protocol, table in FULL["checks"].items():
    print(f"{protocol} split")
    print(table.to_string(index=False))
    print()

Now the two passes side by side.

Most of what this notebook produces cannot legitimately differ between them. Both splits
are built from row counts rather than from data, so every block, every row range and
every check is fixed before a file is opened. Which columns are kept, which are dropped,
which the timing-excluded variant loses, which blocks the scaler was allowed to read, and
how many windows the split implies are all decided the same way in both passes. A
MISMATCH on any of those is a bug in this notebook and not something learned about the
dataset, and nothing from a run with a mismatch should be reported.

What is supposed to differ is listed underneath: how many rows were read, what the scaler
came out as, how many windows were actually built and how much was written. Those are the
whole reason for running twice, and seeing them side by side is what shows the fast pass
did less rather than something else.

In [ ]:
def signature(value, ndigits=10):
    """A comparable form: frames flattened, floats rounded, NaN made equal to itself."""
    if isinstance(value, pd.DataFrame):
        return signature(value.to_dict(orient="records"), ndigits)
    if isinstance(value, pd.Series):
        return signature(value.to_list(), ndigits)
    if isinstance(value, dict):
        return {str(k): signature(v, ndigits) for k, v in value.items()}
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [signature(v, ndigits) for v in value]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return "nan" if not np.isfinite(value) else round(float(value), ndigits)
    if isinstance(value, (int, np.integer)):
        return int(value)
    return value


COMPARISONS = [
    ("features kept", lambda r: FEATURES),
    ("features dropped, and why", lambda r: DROPPED_REASONS),
    ("timing columns still live", lambda r: TIMING_LIVE),
    ("columns the timing slice keeps", lambda r: FEATURES_NO_TIMING),
    ("window and stride", lambda r: [WINDOW, STRIDE]),
    ("array dtype", lambda r: str(np.dtype(ARRAY_DTYPE))),
    ("shipped split, block by block", lambda r: r["shipped_signature"]),
    ("two-tier split, block by block", lambda r: r["block_signature"]),
    ("rows per partition, shipped", lambda r: r["shipped_rows"]),
    ("rows per partition, two-tier", lambda r: r["two_tier_rows"]),
    ("class proportions", lambda r: r["class_proportions"]),
    ("checks and their results", lambda r: r["checks_signature"]),
    ("classes split within one recording", lambda r: r["limitation"]),
    ("rows per class per partition", lambda r: r["rows_by_class"]),
    ("blocks the scaler was allowed", lambda r: r["scaler_signature"]),
    ("scaler read training rows only", lambda r: r["scaler_train_only"]),
    ("windows the split implies", lambda r: r["implied"]),
    ("feature axis of each array written", lambda r: {
        name: line["n_features"] for name, line in r["array_summary"].items()}),
    ("files written", lambda r: sorted(entry["file"] for entry in r["written"])),
]

mismatches = []
print(f"{'value':<38} {'result':<10} fast vs full")
print("-" * 79)
for label, get in COMPARISONS:
    a, b = signature(get(FAST)), signature(get(FULL))
    ok = a == b
    print(f"{label:<38} {'PASS' if ok else 'MISMATCH':<10} " + ("" if ok else "see below"))
    if not ok:
        mismatches.append((label, a, b))

print("-" * 79)
print(f"{len(COMPARISONS) - len(mismatches)} of {len(COMPARISONS)} values agree")
COMPARISON_OK = not mismatches

if mismatches:
    print()
    print("The fast path and the full path disagree on something that cannot legitimately")
    print("differ. That is a bug in this notebook, not a finding. Nothing from this run")
    print("should be reported and nothing it wrote should be read.")
    for label, a, b in mismatches:
        print()
        print(f"  {label}")
        print(f"    fast : {str(a)[:400]}")
        print(f"    full : {str(b)[:400]}")

print()
print("not compared, because these are the values the two passes are supposed to disagree on")
print()
print(f"{'':<38}{'fast':>20}{'full':>20}")
rows = [
    ("windows per block", f"{WINDOWS_PER_BLOCK['fast']}", "uncapped"),
    ("rows the scaler read", f"{FAST['scaler_rows']:,}", f"{FULL['scaler_rows']:,}"),
    ("scaler time", f"{FAST['scaler_seconds']:.1f}s", f"{FULL['scaler_seconds']:.1f}s"),
]
for name in sp.PARTITION_ORDER:
    rows.append((
        f"rows written, {name}",
        f"{FAST['rows_built'][name]:,}",
        f"{FULL['rows_built'][name]:,}",
    ))
    rows.append((
        f"windows written, {name}",
        f"{FAST['array_summary'][f'sequences_{name}']['shape'][0]:,}",
        f"{FULL['array_summary'][f'sequences_{name}']['shape'][0]:,}",
    ))
rows.append(("bytes written", human(FAST["bytes_written"]), human(FULL["bytes_written"])))
rows.append(("time", f"{FAST['elapsed_s']:.0f}s", f"{FULL['elapsed_s']:.0f}s"))
for label, a, b in rows:
    print(f"{label:<38}{a:>20}{b:>20}")

print()
print("The first three features, as each pass fitted them:")
print()
merged = FAST["scaler_statistics"].head(3).merge(
    FULL["scaler_statistics"].head(3), on="feature", suffixes=("_fast", "_full")
)
print(merged[["feature", "mean_fast", "mean_full", "scale_fast", "scale_full"]].to_string(index=False))
print()
print(f"Those two columns are not supposed to match. The fast pass fitted on "
      f"{FAST['scaler_rows']:,} rows, the head of each training block, and the full pass on "
      f"{FULL['scaler_rows']:,},")
print("all of them. Only the second one built the arrays the later notebooks read.")

The entry for `RESULTS_LEDGER.md`, ready to paste. It reports the full pass only. If the
two passes disagreed on anything they should have agreed on, or if any check failed, the
entry says so at the top and the run is not a reference run.

In [ ]:
full_checks = pd.concat(FULL["checks"].values())
checks_passed = bool((full_checks["result"] == "PASS").all())

if not COMPARISON_OK:
    status = "DO NOT ENTER, the fast and full passes disagree and the notebook has a bug"
elif not checks_passed:
    status = "DO NOT ENTER, a split check failed"
elif sum(FULL["rows_built"].values()) != TOTAL_ROWS:
    status = (
        f"DO NOT ENTER, {sum(FULL['rows_built'].values()):,} rows were written and the "
        f"dataset holds {TOTAL_ROWS:,}"
    )
elif GIT_DIRTY:
    status = "reference run, working tree dirty"
else:
    status = "reference run"

sequences_written = {
    name: FULL["array_summary"][f"sequences_{name}"]["shape"][0]
    for name in sp.PARTITION_ORDER
}
smallest = FULL["sequence_table"].min().min()
smallest_class = FULL["sequence_table"].min(axis=1).idxmin()

ledger = f'''
### NB04 — preprocessing and splits ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| pass reported | full, every row of every file read once |
| fast/full agreement | {"all {} compared values agree".format(len(COMPARISONS)) if COMPARISON_OK else "MISMATCH, see the comparison cell"} |
| split checks | {"all {} passed".format(len(full_checks)) if checks_passed else "FAILED, see the checks table"} |
| status | {status} |
| runtime | fast {FAST["elapsed_s"]:.0f}s, full {FULL["elapsed_s"]:.0f}s |
| files read | {len(ALL_FILES)}, {TOTAL_ROWS:,} rows |
| features | {len(FEATURES)}, after dropping {", ".join(DROPPED_REASONS)} |
| timing-excluded slice | defined in the manifest, not written: drop {", ".join(TIMING_LIVE)}, {len(FEATURES_NO_TIMING)} features remain |
| split protocol | two-tier, {RATIOS[0]:.0%}/{RATIOS[1]:.0%}/{RATIOS[2]:.0%}, whole recordings held out for {len(TIER_A)} classes, contiguous blocks for {len(TIER_B)} |
| rows per partition | train {FULL["two_tier_rows"]["train"]:,} ({FULL["two_tier_rows"]["train"] / TOTAL_ROWS:.1%}), val {FULL["two_tier_rows"]["val"]:,} ({FULL["two_tier_rows"]["val"] / TOTAL_ROWS:.1%}), test {FULL["two_tier_rows"]["test"]:,} ({FULL["two_tier_rows"]["test"] / TOTAL_ROWS:.1%}) |
| recordings per partition | train {FULL["manifest"]["split"]["recordings_per_partition"]["train"]}, val {FULL["manifest"]["split"]["recordings_per_partition"]["val"]}, test {FULL["manifest"]["split"]["recordings_per_partition"]["test"]} |
| window, stride | {WINDOW} records, {STRIDE} records, never crossing a file boundary |
| sequences | train {sequences_written["train"]:,}, val {sequences_written["val"]:,}, test {sequences_written["test"]:,} |
| smallest class in sequences | {smallest_class}, {int(smallest):,} in its smallest partition |
| scaler | StandardScaler, fitted on {FULL["scaler_rows"]:,} training rows from {len(FULL["scaler_blocks"])} blocks, no validation or test row read |
| shipped split | saved as file lists and row ranges only, train {FULL["shipped_rows"]["train"]:,} rows, test {FULL["shipped_rows"]["test"]:,}; NB05 reads those CSVs itself and fits its own scaler, so the baseline is reproduced as published |
| artefacts | {FULL["out_dir"]}, {len(FULL["written"])} files, {human(FULL["bytes_written"])} |

Stated limitation: {FULL["limitation"]["statement"]} Those classes are {FULL["limitation"]["share_of_rows"]:.2%} of the rows and {FULL["limitation"]["share_of_test_partition"]:.2%} of the test partition.
'''

print(ledger)